# File Handling (with Exception Handling)

**Session 06 (2026-10-24) · Priority 1 · Notebook 3 of 6**

Variables disappear when a program stops. Files let data **survive**: saved settings, logs, reports, data sets. This notebook combines what you learned in [01_exception_handling.ipynb](01_exception_handling.ipynb) and [02_os_and_paths.ipynb](02_os_and_paths.ipynb).

What you'll learn:

1. What happens behind the scenes when you use a file
2. `open()` and file modes
3. Writing a file
4. The `with` statement
5. Reading a file
6. Appending to a file
7. File handling + exception handling
8. Structured data: CSV-style text and JSON
9. Practice

**Setup.** Every file in this notebook goes into a `practice_files` folder next to the notebook. It's git-ignored, so feel free to experiment.

In [ ]:
import os

BASE_DIR = "practice_files"
os.makedirs(BASE_DIR, exist_ok=True)

notes_path = os.path.join(BASE_DIR, "notes.txt")
print(os.path.abspath(notes_path))

---
## 1. Behind the scenes

Working with a file always has three steps: **open → read or write → close**.

1. **Open.** `open()` asks the operating system for the file. The OS checks that it exists (when reading) and that you're allowed to use it, then hands Python a **file handle**.
2. **File object.** Python wraps the handle in a **file object** with methods like `read()` and `write()`. It also keeps a **cursor**: the position in the file, which moves forward as you read or write.
3. **Buffer.** Writes go into a **buffer** in memory first and reach the disk in chunks, because the disk is slow.
4. **Close.** `close()` **flushes** the buffer to disk and gives the handle back to the OS.

If you forget to close a file, the last writes can be lost, the file can stay locked (especially on Windows), and the OS limits how many files one program can have open at once.

---
## 2. `open()` and file modes

**Syntax:** `open(file, mode="r", encoding=None)`
**Returns:** a file object.

| Mode | Name | If the file exists | If it doesn't exist | Cursor starts at |
|---|---|---|---|---|
| `"r"` | read (the default) | reads it | `FileNotFoundError` | the start |
| `"w"` | write | **erases it first** | creates it | the start |
| `"a"` | append | keeps the content | creates it | the end |
| `"x"` | exclusive create | `FileExistsError` | creates it | the start |
| `"r+"` | read and write | reads and writes | `FileNotFoundError` | the start |

- Add `"b"` for **binary** files such as images and PDFs: `"rb"`, `"wb"`. Text mode (the default) works with `str`; binary mode works with `bytes`.
- **Always pass `encoding="utf-8"` for text files.** Without it, Python uses the operating system's default, which differs between machines, so a file containing ₹, é or emoji may not read correctly on someone else's computer.

---
## 3. Writing a file

First the manual way, so you can see each step.

**`f.write(text)`** writes a string. **Returns** the number of characters written. It does **not** add a new line; you add `"\n"` yourself.

In [ ]:
f = open(notes_path, "w", encoding="utf-8")      # 1. open: creates or erases the file
count = f.write("Line 1: Exception handling\n")  # 2. write
f.write("Line 2: OS library\n")
f.write("Line 3: File handling\n")
f.close()                                        # 3. close: flush to disk

print("characters in the first write:", count)

**The problem with the manual way:** if an error happens between `open()` and `close()`, `close()` never runs. The `with` statement solves this.

---
## 4. The `with` statement (use this every time)

**What it is:** opens a file and **guarantees it's closed** when the indented block ends, even if an error happens inside the block.

**Behind the scenes:** a file object is a **context manager**. `with` calls its setup code on entry and its clean-up code (which closes the file) on exit. It's a shorter way of writing `try` / `finally`:

```python
f = open(path, "w", encoding="utf-8")
try:
    f.write("hello")
finally:
    f.close()          # always runs
```

**Syntax:**

```python
with open(path, mode, encoding="utf-8") as f:
    ...                # use f here
# f is closed here
```

In [ ]:
with open(notes_path, "w", encoding="utf-8") as f:
    f.write("Line 1: Exception handling\n")
    f.write("Line 2: OS library\n")
    f.write("Line 3: File handling\n")
    print("closed inside the block? ", f.closed)

print("closed after the block?  ", f.closed)

---
## 5. Reading a file

| Method | Returns | Good for |
|---|---|---|
| `f.read()` | the whole file as one string | small files |
| `f.readline()` | the next line, including its `\n`; `""` at the end of the file | reading one line at a time by hand |
| `f.readlines()` | a list of every line, each ending in `\n` | small files you want as a list |
| `for line in f:` | one line per loop | **large files**: only one line is in memory at a time |

In [ ]:
with open(notes_path, "r", encoding="utf-8") as f:
    content = f.read()

print(content)
print(type(content), len(content), "characters")

`repr()` shows a string with its quotes and hidden characters, so you can see the `\n` at the end of each line:

In [ ]:
with open(notes_path, encoding="utf-8") as f:   # "r" is the default mode
    first = f.readline()
    second = f.readline()

print(repr(first))
print(repr(second))

In [ ]:
with open(notes_path, encoding="utf-8") as f:
    lines = f.readlines()

print(lines)

Looping over the file is the most common way to read it. Use `strip()` to remove the `\n` from each line:

In [ ]:
with open(notes_path, encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        print(line_number, line.strip())

**The cursor in action.** After `read()`, the cursor is at the end, so a second `read()` returns nothing. `f.seek(0)` moves the cursor back to the start.

In [ ]:
with open(notes_path, encoding="utf-8") as f:
    print(repr(f.read(10)))      # read(10) reads just 10 characters
    print(repr(f.read(10)))      # carries on from where the cursor stopped
    f.read()                     # read the rest: the cursor is now at the end
    print(repr(f.read()))        # nothing left: ''
    f.seek(0)                    # back to the start
    print(repr(f.read(10)))

---
## 6. Appending

`"a"` mode adds to the **end** of the file and keeps what's already there:

In [ ]:
with open(notes_path, "a", encoding="utf-8") as f:
    f.write("Line 4: OOP\n")

with open(notes_path, encoding="utf-8") as f:
    print(f.read())

Compare with `"w"`, which **erases** the file every time it's opened:

In [ ]:
demo_path = os.path.join(BASE_DIR, "overwrite_demo.txt")

with open(demo_path, "w", encoding="utf-8") as f:
    f.write("first version\n")

with open(demo_path, "w", encoding="utf-8") as f:   # "w" again: the old content is gone
    f.write("second version\n")

with open(demo_path, encoding="utf-8") as f:
    print(f.read())

---
## 7. File handling + exception handling

Files depend on things outside your program: the file might not exist, might be locked, or might contain bad data. That makes file code the most common place for exceptions.

| Exception | Typical cause |
|---|---|
| `FileNotFoundError` | reading a file or folder that doesn't exist |
| `FileExistsError` | using `"x"` mode on a file that already exists |
| `PermissionError` | you don't have permission to read or write the file |
| `IsADirectoryError` | passing a folder to `open()` (on Windows this is a `PermissionError`) |
| `UnicodeDecodeError` | reading a file with the wrong encoding, or a binary file in text mode |
| `ValueError` | the content has the wrong format, e.g. `int("abc")` on a line |

The first four are all children of `OSError`, so `except OSError` catches any of them.

### 7.1 Reading a file that might not exist

In [ ]:
def read_file(path):
    try:
        with open(path, encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        print(f"File not found: {path}")
        return None

print(read_file(notes_path))
print(read_file(os.path.join(BASE_DIR, "missing.txt")))

**Check first, or just try?** There are two styles:

- **LBYL** ("look before you leap"): check with `os.path.exists()` first.
- **EAFP** ("easier to ask forgiveness than permission"): just try, and handle the exception.

Python usually prefers **EAFP**. The file could be deleted between your check and your `open()`, and a `try` handles every kind of failure, not just "missing".

In [ ]:
missing = os.path.join(BASE_DIR, "missing.txt")

# LBYL
if os.path.exists(missing):
    with open(missing, encoding="utf-8") as f:
        print(f.read())
else:
    print("LBYL: the file does not exist")

# EAFP
try:
    with open(missing, encoding="utf-8") as f:
        print(f.read())
except FileNotFoundError:
    print("EAFP: the file does not exist")

### 7.2 Handling several file errors

In [ ]:
def safe_read(path):
    try:
        with open(path, encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        print(f"[{path}] does not exist")
    except IsADirectoryError:
        print(f"[{path}] is a folder, not a file")
    except PermissionError:
        print(f"[{path}] permission denied")
    except UnicodeDecodeError:
        print(f"[{path}] is not a UTF-8 text file")
    return None

safe_read("no_such_file.txt")
safe_read(BASE_DIR)                  # a folder, not a file

A binary file read in text mode raises `UnicodeDecodeError`. `"wb"` writes raw **bytes** instead of text:

In [ ]:
binary_path = os.path.join(BASE_DIR, "binary.bin")
with open(binary_path, "wb") as f:
    f.write(bytes([0xFF, 0xFE, 0x00, 0x80]))   # these bytes are not valid UTF-8 text

safe_read(binary_path)

### 7.3 Never overwrite by accident: `"x"` mode

`"x"` creates a file only if it doesn't already exist, so you can't wipe out existing data by mistake.

In [ ]:
report_path = os.path.join(BASE_DIR, "report.txt")
if os.path.exists(report_path):
    os.remove(report_path)            # start fresh, so this cell gives the same result every run

for attempt in [1, 2]:
    try:
        with open(report_path, "x", encoding="utf-8") as f:
            f.write("monthly report\n")
        print(f"attempt {attempt}: created the report")
    except FileExistsError:
        print(f"attempt {attempt}: the report already exists, not overwriting it")

### 7.4 Bad data inside a file

Real files contain mistakes. Handle errors **line by line**, so one bad line doesn't stop the whole file from loading.

First, create a file with some bad lines in it:

In [ ]:
marks_path = os.path.join(BASE_DIR, "marks.txt")
with open(marks_path, "w", encoding="utf-8") as f:
    f.write("asha,92\n")
    f.write("ravi,85\n")
    f.write("monal,abc\n")     # the mark isn't a number
    f.write("alex\n")          # the mark is missing
    f.write("srinu,78\n")

This function uses **two levels** of `try`:

- the **outer** `try` handles the file itself (it might be missing)
- the **inner** `try` handles each line (it might be badly formatted)

One `except ValueError` catches both kinds of bad line: `name, mark = ...` raises `ValueError` when a line doesn't split into exactly two parts, and `int(mark)` raises `ValueError` when the mark isn't a number.

In [ ]:
def load_marks(path):
    marks = {}
    try:
        with open(path, encoding="utf-8") as f:
            for line_number, line in enumerate(f, start=1):
                try:
                    name, mark = line.strip().split(",")
                    marks[name] = int(mark)
                except ValueError:
                    print(f"  skipping line {line_number}: '{line.strip()}'")
    except FileNotFoundError:
        print(f"{path} not found")
    return marks

marks = load_marks(marks_path)
print(marks)

In [ ]:
print(load_marks(os.path.join(BASE_DIR, "no_marks.txt")))   # missing file: an empty dict

---
## 8. Structured data

### 8.1 CSV-style text

**CSV** (comma-separated values) stores one record per line, with fields separated by commas. The first line is usually a **header** naming the columns.

In [ ]:
students = {"asha": 92, "ravi": 85, "srinu": 78}

csv_path = os.path.join(BASE_DIR, "students.csv")
with open(csv_path, "w", encoding="utf-8") as f:
    f.write("name,mark\n")                    # header row
    for name, mark in students.items():
        f.write(f"{name},{mark}\n")

with open(csv_path, encoding="utf-8") as f:
    print(f.read())

Splitting on commas is fine for simple data. For real CSV files, where a field can contain a comma inside quotes, use the built-in `csv` module.

### 8.2 JSON

**What it is:** JSON is a text format that looks almost like Python dicts and lists. It's the standard format for web APIs and config files.

**Behind the scenes:** `json.dump()` turns Python objects into JSON text and writes it (**serializing**). `json.load()` reads the text and builds the Python objects again (**deserializing**).

| Python | JSON |
|---|---|
| `dict` | object `{...}` |
| `list`, `tuple` | array `[...]` |
| `str` | string (**double quotes only**) |
| `int`, `float` | number |
| `True` / `False` | `true` / `false` |
| `None` | `null` |

**Syntax:** `json.dump(obj, file, indent=None)` writes; `json.load(file)` reads.
**Returns:** `dump` returns `None`; `load` returns the Python object.

In [ ]:
import json

config = {
    "batch": "FDE v1.0",
    "students": 25,
    "topics": ["exceptions", "os", "files"],
    "active": True,
}
config_path = os.path.join(BASE_DIR, "config.json")

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)          # indent makes the file easy to read

with open(config_path, encoding="utf-8") as f:
    print(f.read())                          # the raw JSON text: note true, not True

In [ ]:
def load_json(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"{path} not found")
    except json.JSONDecodeError as e:
        print(f"{path} is not valid JSON: {e}")
    return None

loaded = load_json(config_path)
print(loaded, type(loaded))
print(loaded["topics"])

A file that isn't valid JSON raises `json.JSONDecodeError`. Here the mistake is single quotes, which JSON doesn't allow:

In [ ]:
broken_path = os.path.join(BASE_DIR, "broken.json")
with open(broken_path, "w", encoding="utf-8") as f:
    f.write("{'batch': 'FDE'}")

print(load_json(broken_path))

---
## 9. Practice

**1.** Write `count_lines(path)` that returns the number of lines in a file. If the file is missing, print a message and return `0`.

In [ ]:
# Exercise 1

**2.** Write `append_log(path, message)` that appends one line per call. Call it three times, then print the whole file.

In [ ]:
# Exercise 2

**3.** Write `copy_file(src, dst)` that copies a text file. It must refuse to overwrite `dst` (use `"x"` mode) and must handle a missing `src`, with a different message for each problem.

In [ ]:
# Exercise 3

**4.** Extend `load_marks` so it also skips marks outside 0 to 100, and returns the **average** of the valid marks as well as the dict.

In [ ]:
# Exercise 4

**5.** Save a dict of three topics with a rating from 1 to 5 to a JSON file, load it back, and print the topic with the highest rating.

In [ ]:
# Exercise 5